In [1]:
"""
TOP1模型 (Gp_GP_Matern_0.5_K6_Comb17) Pair Plot 可视化
配色更新版：12色蓝红渐变色板（去掉原两端最深色）
Train: #6BAED6（新第3色蓝）/ Test: #FB6A4A（新第10色红）/ Fit Line: #ab3f99
"""

import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from scipy.stats import pearsonr
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

# ====================== 全局字体设置 ======================
matplotlib.rcParams['font.family']        = 'Arial'
matplotlib.rcParams['pdf.fonttype']       = 42
matplotlib.rcParams['ps.fonttype']        = 42
matplotlib.rcParams['axes.unicode_minus'] = False

# ====================== 1. 路径配置 ======================
DATA_FILE = (
    r"D:\博一下\第一章主动学习"
    r"\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx"
)
PKL_FILE = (
    r"D:\博一下\第一章主动学习\高斯过程组_结果_v3.5-Extended"
    r"\top30_models\model_objects"
    r"\rank01_Gp_GP_Matern_0.5_K6_Comb17_model.pkl"
)
OUTPUT_DIR = r"D:\博二上\模型分析可视化\pair plot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====================== 2. 加载pkl ======================
print("加载TOP1模型pkl...")
with open(PKL_FILE, 'rb') as f:
    model_data = pickle.load(f)

gp_model      = model_data['model']
scaler        = model_data['scaler']
feature_names = model_data['features']

print(f"模型类型  : {model_data['model_name']}")
print(f"特征列表  : {feature_names}")

# ====================== 3. 特征简写映射 ======================
rename_dict = {
    'deltaP_热导率W/(mk)':           'PT1d',
    'x':                             'SG2',
    'Ec':                            'JX7',
    'Ω':                             'JX3',
    'deltaP_E13 Electron affinity':  'AE4d',
    'ΔSmix':                         'JA2',
}
display_features = [rename_dict.get(f, f) for f in feature_names]
print(f"显示名称  : {display_features}")

# ====================== 4. 读取数据 ======================
print("\n读取数据...")
df = pd.read_excel(DATA_FILE)

kq_col = None
for c in ['KQ', 'kq', 'K_Q', 'KQ值', 'KQ(MPa·m^1/2)']:
    if c in df.columns:
        kq_col = c
        break
if kq_col is None:
    raise ValueError("找不到KQ列，请检查Excel文件列名")

df_valid = df[feature_names + [kq_col]].dropna()
df_valid = df_valid[df_valid[kq_col] > 0].reset_index(drop=True)
print(f"有效样本数: {len(df_valid)}")

X_all = df_valid[feature_names].values
y_all = df_valid[kq_col].values

# ====================== 5. 重现程序1的训练/测试集划分 ======================
def create_stratify_bins(y, n_bins=5):
    percentiles = np.linspace(0, 100, n_bins + 1)
    bin_edges   = np.percentile(y, percentiles)
    bin_edges[0]  = -np.inf
    bin_edges[-1] =  np.inf
    return np.digitize(y, bin_edges) - 1

stratify_labels = create_stratify_bins(y_all, n_bins=5)
indices_all     = np.arange(len(y_all))

train_indices, test_indices, _, _ = train_test_split(
    indices_all, indices_all,
    test_size=0.20,
    stratify=stratify_labels,
    random_state=2023
)
print(f"训练集: {len(train_indices)}  测试集: {len(test_indices)}")

# ====================== 6. 直接调用已训练模型预测 ======================
X_train_scaled = scaler.transform(X_all[train_indices])
X_test_scaled  = scaler.transform(X_all[test_indices])

pred_train = gp_model.predict(X_train_scaled)
pred_test  = gp_model.predict(X_test_scaled)

r2_tr   = r2_score(y_all[train_indices],  pred_train)
rmse_tr = np.sqrt(mean_squared_error(y_all[train_indices], pred_train))
mae_tr  = mean_absolute_error(y_all[train_indices], pred_train)
r2_te   = r2_score(y_all[test_indices],   pred_test)
rmse_te = np.sqrt(mean_squared_error(y_all[test_indices], pred_test))
mae_te  = mean_absolute_error(y_all[test_indices], pred_test)

print(f"\nTrain  R²={r2_tr:.4f}  RMSE={rmse_tr:.4f}  MAE={mae_tr:.4f}")
print(f"Test   R²={r2_te:.4f}  RMSE={rmse_te:.4f}  MAE={mae_te:.4f}")

# ====================== 7. Pair Plot 数据准备 ======================
data_all   = X_all
data_train = X_all[train_indices]
data_test  = X_all[test_indices]
n_cols     = len(feature_names)   # 6

# SG2(x) 对应列索引，y轴保留两位小数
sg2_idx = feature_names.index('x')

# ====================== 8. 配色配置 ======================

# ── 散点颜色 ─────────────────────────────────────────────
# 新12色中对称选取：新第3色(蓝) + 新第10色(红)
COLOR_TRAIN    = '#6BAED6'   # 新第3色  蓝（对称适中）
COLOR_TEST     = '#FB6A4A'   # 新第10色 红（对称适中）

# ── Fit Line 颜色 ────────────────────────────────────────
FIT_LINE_COLOR = '#ab3f99'   # 紫色

# ── 上三角色条：12色蓝红渐变 ─────────────────────────────
# 去掉原14色两端最深色（原#08519C和原#A50F15）
# 零点精确位于新第6色与新第7色之间（节点 = 5.5/11 = 0.5）
_colors_12 = [
    '#2171B5',   # 新第1  蓝      → r = -1.00
    '#4292C6',   # 新第2  中蓝    → r ≈ -0.82
    '#6BAED6',   # 新第3  浅中蓝  → r ≈ -0.64  ← Train色
    '#9ECAE1',   # 新第4  浅蓝    → r ≈ -0.45
    '#C6DBEF',   # 新第5  极浅蓝  → r ≈ -0.27
    '#DEEBF7',   # 新第6  近白蓝  → r ≈ -0.09
    '#FEE0D2',   # 新第7  近白红  → r ≈ +0.09
    '#FCBBA1',   # 新第8  极浅红  → r ≈ +0.27
    '#FC9272',   # 新第9  浅红    → r ≈ +0.45
    '#FB6A4A',   # 新第10 中红    → r ≈ +0.64  ← Test色
    '#EF3B2C',   # 新第11 红      → r ≈ +0.82
    '#CB181D',   # 新第12 深红    → r = +1.00
]
_nodes_12  = np.linspace(0.0, 1.0, len(_colors_12))
CMAP_CORR  = LinearSegmentedColormap.from_list(
    'corr_12', list(zip(_nodes_12, _colors_12))
)

# 零点在色图中的归一化位置（新第6/7色之间）
_vcenter_node = 5.5 / 11.0   # = 0.500

def corr_bg_color(r):
    """r ∈ [-1, 1]  →  12色色图RGBA"""
    if r >= 0:
        t = _vcenter_node + r * (1.0 - _vcenter_node)
    else:
        t = _vcenter_node + r * _vcenter_node
    return CMAP_CORR(np.clip(t, 0.0, 1.0))

def text_color_for_bg(r):
    """|r| > 0.50 深色背景用白字，否则用深色字"""
    return 'white' if abs(r) > 0.50 else '#333333'

# ====================== 9. 辅助函数 ======================
def sig_stars(p):
    if   p < 0.001: return '***'
    elif p < 0.01:  return '**'
    elif p < 0.05:  return '*'
    else:           return ''

# ====================== 10. 绘图主体 ======================
print("\n绘制Pair Plot...")

fig = plt.figure(figsize=(14, 14))

left_m    = 0.08
plot_size = 0.69
right_m   = left_m + plot_size
bottom_m  = 0.08
top_m     = bottom_m + plot_size

gs_main = gridspec.GridSpec(
    n_cols, n_cols,
    left=left_m,   right=right_m,
    top=top_m,     bottom=bottom_m,
    hspace=0.04,   wspace=0.04,
    height_ratios=[1] * n_cols,
    width_ratios=[1]  * n_cols,
)

axes = np.empty((n_cols, n_cols), dtype=object)
for r in range(n_cols):
    for c in range(n_cols):
        axes[r, c] = fig.add_subplot(gs_main[r, c])

for row in range(n_cols):
    for col in range(n_cols):
        ax      = axes[row, col]
        x_all_p = data_all[:,   col]
        y_all_p = data_all[:,   row]
        x_tr    = data_train[:, col]
        y_tr    = data_train[:, row]
        x_te    = data_test[:,  col]
        y_te    = data_test[:,  row]

        ax.set_box_aspect(1)

        # ── 对角线：直方图 ────────────────────────────────
        if row == col:
            ax.hist(x_tr, bins=8, color=COLOR_TRAIN, alpha=0.65,
                    edgecolor='white', linewidth=0.5, density=False)
            ax.hist(x_te, bins=8, color=COLOR_TEST,  alpha=0.65,
                    edgecolor='white', linewidth=0.5, density=False)
            ax.set_facecolor('#FAFAFA')
            for spine in ax.spines.values():
                spine.set_linewidth(0.6)
                spine.set_color('#CCCCCC')

        # ── 上三角：Pearson r + 显著性星号 ───────────────
        elif col > row:
            r_val, p_val = pearsonr(x_all_p, y_all_p)
            stars        = sig_stars(p_val)
            bg           = corr_bg_color(r_val)
            ax.set_facecolor(bg)

            ax.text(0.5, 0.55, f'r = {r_val:.2f}',
                    transform=ax.transAxes,
                    ha='center', va='center',
                    fontsize=15, fontweight='bold',
                    color=text_color_for_bg(r_val),
                    fontfamily='Arial')

            if stars:
                ax.text(0.5, 0.28, stars,
                        transform=ax.transAxes,
                        ha='center', va='center',
                        fontsize=14,
                        color=text_color_for_bg(r_val),
                        fontfamily='Arial')

            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(0.4)
                spine.set_color('#CCCCCC')

        # ── 下三角：散点 + 拟合线 + 95%CI ───────────────
        else:
            ax.scatter(x_tr, y_tr, s=20, color=COLOR_TRAIN,
                       alpha=0.7, linewidths=0,
                       edgecolors='none', zorder=3)
            ax.scatter(x_te, y_te, s=28, color=COLOR_TEST,
                       alpha=0.85, linewidths=0.4,
                       edgecolors='white', zorder=4)

            if len(x_tr) > 2:
                slope, intercept, _, _, se = stats.linregress(x_tr, y_tr)
                x_line = np.linspace(x_tr.min(), x_tr.max(), 100)
                y_line = slope * x_line + intercept

                ax.plot(x_line, y_line,
                        color=FIT_LINE_COLOR,
                        linewidth=1.2, linestyle='--',
                        alpha=0.9, zorder=2)

                n_tr   = len(x_tr)
                x_mean = np.mean(x_tr)
                se_ln  = se * np.sqrt(
                    1 / n_tr +
                    (x_line - x_mean) ** 2 /
                    np.sum((x_tr - x_mean) ** 2)
                )
                t_val  = stats.t.ppf(0.975, df=n_tr - 2)
                ax.fill_between(
                    x_line,
                    y_line - t_val * se_ln,
                    y_line + t_val * se_ln,
                    alpha=0.12, color='#888888', zorder=1
                )

            ax.set_facecolor('#FAFAFA')
            for spine in ax.spines.values():
                spine.set_linewidth(0.6)
                spine.set_color('#CCCCCC')

        # ── 刻度样式 ─────────────────────────────────────
        ax.tick_params(axis='both', labelsize=7, length=2,
                       color='#999999', labelcolor='#555555',
                       pad=2, direction='in')

        # SG2(x) 作为纵坐标时 y轴保留两位小数
        if row == sg2_idx and col != sg2_idx:
            ax.yaxis.set_major_formatter(
                matplotlib.ticker.FormatStrFormatter('%.2f')
            )

        # ── 轴标签 ───────────────────────────────────────
        if row == n_cols - 1:
            ax.set_xlabel(display_features[col], fontsize=11,
                          fontweight='bold', labelpad=4,
                          color='#333333', fontfamily='Arial')
        else:
            ax.set_xticklabels([])

        if col == 0:
            ax.set_ylabel(display_features[row], fontsize=11,
                          fontweight='bold', labelpad=4,
                          color='#333333', rotation=90,
                          fontfamily='Arial')
        else:
            ax.set_yticklabels([])

# ====================== 11. 图例与色条 ======================

# ── Train 图例 ────────────────────────────────────────────
lg1 = fig.add_axes([0.79, 0.70, 0.18, 0.04])
lg1.set_axis_off()
lg1.scatter([0.05], [0.5], s=80, color=COLOR_TRAIN,
            alpha=0.7, edgecolors='none',
            transform=lg1.transAxes, clip_on=False)
lg1.text(0.18, 0.5, 'Train',
         transform=lg1.transAxes, va='center', ha='left',
         fontsize=14, color='#333333', fontweight='bold',
         fontfamily='Arial')

# ── Test 图例 ─────────────────────────────────────────────
lg2 = fig.add_axes([0.79, 0.66, 0.18, 0.04])
lg2.set_axis_off()
lg2.scatter([0.05], [0.5], s=100, color=COLOR_TEST,
            alpha=0.85, edgecolors='white', linewidths=0.8,
            transform=lg2.transAxes, clip_on=False)
lg2.text(0.18, 0.5, 'Test',
         transform=lg2.transAxes, va='center', ha='left',
         fontsize=14, color='#333333', fontweight='bold',
         fontfamily='Arial')

# ── Fit line 图例 ─────────────────────────────────────────
lg3 = fig.add_axes([0.79, 0.58, 0.18, 0.05])
lg3.set_axis_off()
lg3.plot([0.02, 0.12], [0.5, 0.5],
         color=FIT_LINE_COLOR,
         linewidth=1.5, linestyle='--',
         transform=lg3.transAxes, clip_on=False)
lg3.text(0.02, 0.70, 'Fit line',
         transform=lg3.transAxes, va='center', ha='left',
         fontsize=14, color='#333333', fontweight='bold',
         fontfamily='Arial')
lg3.text(0.02, 0.30, '(Train)',
         transform=lg3.transAxes, va='center', ha='left',
         fontsize=14, color='#333333', fontweight='bold',
         fontfamily='Arial')

# ── 95% CI 图例 ───────────────────────────────────────────
lg4 = fig.add_axes([0.79, 0.51, 0.18, 0.05])
lg4.set_axis_off()
lg4.fill_between([0.02, 0.12], [0.35, 0.35], [0.65, 0.65],
                  alpha=0.25, color='#888888',
                  transform=lg4.transAxes, clip_on=False)
lg4.text(0.02, 0.70, '95% CI',
         transform=lg4.transAxes, va='center', ha='left',
         fontsize=14, color='#333333', fontweight='bold',
         fontfamily='Arial')
lg4.text(0.02, 0.30, '(Train)',
         transform=lg4.transAxes, va='center', ha='left',
         fontsize=14, color='#333333', fontweight='bold',
         fontfamily='Arial')

# ── 色条（12色渐变）─────────────────────────────────────
cbar_ax = fig.add_axes([0.795, 0.18, 0.018, 0.20])
norm_cb = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
sm      = plt.cm.ScalarMappable(cmap=CMAP_CORR, norm=norm_cb)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')
cbar.set_label('Pearson r', fontsize=11, fontweight='bold',
               color='#333333', labelpad=8, fontfamily='Arial')
cbar.ax.tick_params(labelsize=10, color='#888888',
                    labelcolor='#555555', direction='in')
cbar.outline.set_edgecolor('#CCCCCC')
cbar.outline.set_linewidth(0.6)
cbar.set_ticks([-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1])

# ── 显著性说明 ────────────────────────────────────────────
sig_ax = fig.add_axes([0.79, 0.08, 0.18, 0.08])
sig_ax.set_axis_off()
sig_ax.text(0.0, 0.75, '* p < 0.05',
            fontsize=10, color='#555555',
            transform=sig_ax.transAxes, va='center',
            fontfamily='Arial')
sig_ax.text(0.0, 0.45, '** p < 0.01',
            fontsize=10, color='#555555',
            transform=sig_ax.transAxes, va='center',
            fontfamily='Arial')
sig_ax.text(0.0, 0.15, '*** p < 0.001',
            fontsize=10, color='#555555',
            transform=sig_ax.transAxes, va='center',
            fontfamily='Arial')

# ====================== 12. 保存图片 ======================
save_path = os.path.join(
    OUTPUT_DIR,
    'TOP1_GP_Matern0.5_K6_Comb17_pairplot_v6.png'
)
fig.savefig(save_path, dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close(fig)
print(f"\n✓ Pair Plot 已保存: {save_path}")

# ====================== 13. 保存原始数据到Excel ======================
df_export = pd.DataFrame(data_all, columns=feature_names)
df_export.rename(columns=rename_dict, inplace=True)
df_export['KQ']      = y_all
df_export['Dataset'] = 'train'
df_export.loc[test_indices, 'Dataset'] = 'test'

excel_path = os.path.join(OUTPUT_DIR, 'TOP1_pairplot_data_v6.xlsx')
df_export.to_excel(excel_path, index=False)
print(f"✓ 数据已保存: {excel_path}")
print("\n程序执行完毕！")

加载TOP1模型pkl...
模型类型  : GP_Matern_0.5
特征列表  : ['deltaP_热导率W/(mk)', 'x', 'Ec', 'Ω', 'deltaP_E13 Electron affinity', 'ΔSmix']
显示名称  : ['PT1d', 'SG2', 'JX7', 'JX3', 'AE4d', 'JA2']

读取数据...
有效样本数: 202
训练集: 161  测试集: 41

Train  R²=0.8748  RMSE=1.0899  MAE=0.8392
Test   R²=0.7598  RMSE=1.5575  MAE=1.2080

绘制Pair Plot...

✓ Pair Plot 已保存: D:\博二上\模型分析可视化\pair plot\TOP1_GP_Matern0.5_K6_Comb17_pairplot_v6.png
✓ 数据已保存: D:\博二上\模型分析可视化\pair plot\TOP1_pairplot_data_v6.xlsx

程序执行完毕！
